# Week 09 — TensorFlow Lite and On-Device Inference
## LeafGuard AI: Running the ML Model Offline on Android

This notebook explores TFLite — how to convert, optimise, and run plant disease models on Android.

**Learning outcomes:**
- Understand why TFLite exists (size, speed, offline use)
- Convert a Keras model to TFLite format
- Apply quantization to reduce model size by 75%
- Simulate the Android TFLite inference pipeline in Python
- Understand benchmarking and performance measurement
- Map the Python workflow to `TFLiteClassifier.java`

---

## Cell 1: Why TFLite?

Full TensorFlow vs TFLite comparison.

In [ ]:
import json

comparison = {
    'Full TensorFlow': {
        'platforms': 'Python, servers, cloud, GPU clusters',
        'use_case': 'Training models, complex preprocessing pipelines',
        'model_format': '.h5 or SavedModel directory',
        'runtime_size': '300-500 MB',
        'inference_speed_cpu': 'Medium',
        'supports_training': True,
        'internet_required': False,
        'ops_supported': 'All TensorFlow operations',
    },
    'TensorFlow Lite': {
        'platforms': 'Android, iOS, Raspberry Pi, microcontrollers',
        'use_case': 'On-device inference in production apps',
        'model_format': '.tflite (FlatBuffer binary)',
        'runtime_size': '1-2 MB (Android SDK included)',
        'inference_speed_cpu': 'Fast (optimised kernels)',
        'supports_training': False,
        'internet_required': False,
        'ops_supported': 'Subset optimised for mobile',
    }
}

print('=== Full TensorFlow vs TFLite ===')
print()

fields = list(next(iter(comparison.values())).keys())
col_w = 35
print(f'{"Feature":<30} {"Full TensorFlow":<col_w} {"TFLite":<col_w}'.replace('col_w', str(col_w)))
print('-' * (30 + col_w * 2 + 2))

for field in fields:
    tf_val = str(comparison['Full TensorFlow'][field])
    tflite_val = str(comparison['TensorFlow Lite'][field])
    print(f'{field.replace("_", " ").title():<30} {tf_val[:col_w-2]:<{col_w}} {tflite_val[:col_w-2]}')

print()
print('LeafGuard AI uses BOTH:')
print('  - Full TF on FastAPI server (cloud mode, larger/more accurate model)')
print('  - TFLite on Android device (offline mode, compact quantised model)')

## Cell 2: Quantization Types and Their Impact

In [ ]:
import numpy as np

print('=== QUANTIZATION: FP32 → INT8 ===')
print()

# Demonstrate float32 vs int8 representation
# A weight value in the neural network
original_weight = 0.73456789

# Float32 representation (4 bytes per value)
fp32_value = np.float32(original_weight)
fp32_bytes = fp32_value.tobytes()
print(f'Float32 (original):')
print(f'  Value: {fp32_value}')
print(f'  Bytes: {fp32_bytes.hex()} ({len(fp32_bytes)} bytes)')
print(f'  Precision: ~7 decimal digits')

print()

# Int8 representation (1 byte per value)
# Scale: map [-1, 1] → [-128, 127]
scale = 1.0 / 127.0
zero_point = 0
int8_value = np.int8(int(original_weight / scale + zero_point))
dequantized = (float(int8_value) - zero_point) * scale

print(f'Int8 (quantized):')
print(f'  Quantized: {int8_value}')
print(f'  Bytes: {int8_value.tobytes().hex()} ({len(int8_value.tobytes())} byte)')
print(f'  Dequantized back: {dequantized:.6f}')
print(f'  Error introduced: {abs(original_weight - dequantized):.6f}')
print(f'  Relative error: {abs(original_weight - dequantized)/original_weight*100:.3f}%')

print()
print('Size comparison for the full plant disease model:')

model_weights = 4_000_000  # ~4M weight values in MobileNetV2

fp32_size_mb = model_weights * 4 / (1024**2)
int8_size_mb = model_weights * 1 / (1024**2)
fp16_size_mb = model_weights * 2 / (1024**2)

print(f'  FP32: {fp32_size_mb:.1f} MB')
print(f'  FP16: {fp16_size_mb:.1f} MB ({(1-fp16_size_mb/fp32_size_mb)*100:.0f}% smaller)')
print(f'  INT8: {int8_size_mb:.1f} MB ({(1-int8_size_mb/fp32_size_mb)*100:.0f}% smaller) ← What TFLite uses')

print()
print('Accuracy impact: INT8 quantization typically loses < 1-2% accuracy')
print('Speed impact: INT8 is 2-4× faster on Android CPU (SIMD optimisation)')

## Cell 3: TFLite Conversion (requires TensorFlow)

In [ ]:
try:
    import tensorflow as tf
    import numpy as np
    import os

    print('=== TFLITE CONVERSION PIPELINE ===')
    print()

    # Build a representative small model
    # (In practice, load the trained plant disease model)
    inputs = tf.keras.Input(shape=(224, 224, 3), name='input_image')
    x = tf.keras.layers.Conv2D(32, 3, activation='relu')(inputs)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    outputs = tf.keras.layers.Dense(38, activation='softmax', name='predictions')(x)
    demo_model = tf.keras.Model(inputs, outputs)

    print(f'Demo model input:  {demo_model.input_shape}')
    print(f'Demo model output: {demo_model.output_shape}')
    print(f'Demo model params: {demo_model.count_params():,}')
    print()

    results = {}

    # ---- FP32 (baseline) ----
    converter = tf.lite.TFLiteConverter.from_keras_model(demo_model)
    tflite_fp32 = converter.convert()
    results['FP32 (no quantization)'] = len(tflite_fp32)

    # ---- FP16 quantization ----
    converter = tf.lite.TFLiteConverter.from_keras_model(demo_model)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    converter.target_spec.supported_types = [tf.float16]
    tflite_fp16 = converter.convert()
    results['FP16 quantization'] = len(tflite_fp16)

    # ---- INT8 dynamic range quantization ----
    converter = tf.lite.TFLiteConverter.from_keras_model(demo_model)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    tflite_int8 = converter.convert()
    results['INT8 dynamic range'] = len(tflite_int8)

    # Display results
    baseline = results['FP32 (no quantization)']
    print('Conversion results:')
    print(f'{"Format":<30} {"Size (KB)":>10} {"Reduction":>12}')
    print('-' * 55)
    for name, size in results.items():
        reduction = f'{(1 - size/baseline)*100:.1f}%' if size != baseline else '-'
        print(f'{name:<30} {size/1024:>10.1f} {reduction:>12}')

    # Verify TFLite FP32 model works
    print()
    print('Running inference with TFLite interpreter...')
    interpreter = tf.lite.Interpreter(model_content=tflite_fp32)
    interpreter.allocate_tensors()

    input_details = interpreter.get_input_details()[0]
    output_details = interpreter.get_output_details()[0]

    # Create test input
    test_input = np.random.rand(1, 224, 224, 3).astype(np.float32)
    interpreter.set_tensor(input_details['index'], test_input)
    interpreter.invoke()
    output = interpreter.get_tensor(output_details['index'])

    print(f'Input shape:  {test_input.shape}')
    print(f'Output shape: {output.shape}')
    print(f'Probabilities sum: {output[0].sum():.6f} (should be 1.0)')
    print(f'Top class index:   {np.argmax(output[0])}')
    print(f'Top confidence:    {output[0].max()*100:.1f}%')

except ImportError:
    print('TensorFlow not installed. Install with: pip install tensorflow')
    print()
    print('The conversion code above shows the complete pipeline.')
    print('Key points to remember:')
    print('  1. converter = tf.lite.TFLiteConverter.from_keras_model(model)')
    print('  2. Set optimizations for quantization')
    print('  3. tflite_model = converter.convert()')
    print('  4. Save as .tflite file to assets/ folder in Android project')

## Cell 4: Android TFLiteClassifier — Python Simulation

The Android `TFLiteClassifier.java` does exactly this in Java.
Understanding it in Python first makes the Java code easy to follow.

In [ ]:
import numpy as np
from PIL import Image
import io
import time

# Simulate what TFLiteClassifier.java does

CLASS_NAMES = [
    'Apple___Apple_scab', 'Apple___Black_rot', 'Apple___Cedar_apple_rust', 'Apple___healthy',
    'Blueberry___healthy', 'Cherry_(including_sour)___Powdery_mildew', 'Cherry_(including_sour)___healthy',
    'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot', 'Corn_(maize)___Common_rust_',
    'Corn_(maize)___Northern_Leaf_Blight', 'Corn_(maize)___healthy', 'Grape___Black_rot',
    'Grape___Esca_(Black_Measles)', 'Grape___Leaf_blight_(Isariopsis_Leaf_Spot)', 'Grape___healthy',
    'Orange___Haunglongbing_(Citrus_greening)', 'Peach___Bacterial_spot', 'Peach___healthy',
    'Pepper,_bell___Bacterial_spot', 'Pepper,_bell___healthy', 'Potato___Early_blight',
    'Potato___Late_blight', 'Potato___healthy', 'Raspberry___healthy', 'Soybean___healthy',
    'Squash___Powdery_mildew', 'Strawberry___Leaf_scorch', 'Strawberry___healthy',
    'Tomato___Bacterial_spot', 'Tomato___Early_blight', 'Tomato___Late_blight',
    'Tomato___Leaf_Mold', 'Tomato___Septoria_leaf_spot',
    'Tomato___Spider_mites Two-spotted_spider_mite', 'Tomato___Target_Spot',
    'Tomato___Tomato_Yellow_Leaf_Curl_Virus', 'Tomato___Tomato_mosaic_virus', 'Tomato___healthy'
]

class TFLiteClassifierSimulator:
    """
    Python simulation of TFLiteClassifier.java.
    Shows the exact same steps the Android app performs.
    """

    def __init__(self, class_names):
        self.class_names = class_names
        self.model_loaded = False
        print('[TFLiteClassifier] Initialized (model not yet loaded)')

    def initialize(self):
        """
        Step 1: Load model from assets.
        Java: MappedByteBuffer modelBuffer = loadModelFile(context, "plant_disease.tflite");
              interpreter = new Interpreter(modelBuffer);
        """
        # In Python, we simulate model loading
        # In real Android, this loads the .tflite file from assets/
        print('[TFLiteClassifier] Loading model from assets/plant_disease.tflite...')
        time.sleep(0.05)  # Simulate 50ms load time
        self.model_loaded = True
        print(f'[TFLiteClassifier] Model loaded! Classes: {len(self.class_names)}')

    def preprocess_bitmap(self, bitmap: Image.Image) -> np.ndarray:
        """
        Step 2: Preprocess Android Bitmap to float tensor.

        Java equivalent:
        private float[][][][] preprocessBitmap(Bitmap bitmap) {
            Bitmap resized = Bitmap.createScaledBitmap(bitmap, 224, 224, true);
            float[][][][] input = new float[1][224][224][3];
            for (int y = 0; y < 224; y++) {
                for (int x = 0; x < 224; x++) {
                    int pixel = resized.getPixel(x, y);
                    input[0][y][x][0] = ((pixel >> 16) & 0xFF) / 255.0f;  // R
                    input[0][y][x][1] = ((pixel >> 8)  & 0xFF) / 255.0f;  // G
                    input[0][y][x][2] = (pixel          & 0xFF) / 255.0f;  // B
                }
            }
            return input;
        }
        """
        # Resize to 224×224
        resized = bitmap.resize((224, 224), Image.LANCZOS)

        # Convert to RGB (handles RGBA images from camera)
        resized = resized.convert('RGB')

        # Convert to float32 array, normalise
        img_array = np.array(resized, dtype=np.float32) / 255.0  # [H, W, 3]

        # Add batch dimension: [H, W, 3] → [1, H, W, 3]
        img_array = np.expand_dims(img_array, axis=0)

        return img_array

    def run_inference(self, input_tensor: np.ndarray) -> np.ndarray:
        """
        Step 3: Run TFLite interpreter.

        Java equivalent:
        float[][][][] input = preprocessBitmap(bitmap);
        float[][] output = new float[1][labels.size()];
        interpreter.run(input, output);
        """
        if not self.model_loaded:
            raise RuntimeError('Model not initialized! Call initialize() first.')

        # Simulate model inference with random softmax output
        # Real TFLite: interpreter.run(input, output)
        np.random.seed(29)  # Deterministic for demo (class 29 = Tomato Early Blight)
        logits = np.random.randn(len(self.class_names)) * 2
        logits[29] = 5.2  # Boost Early Blight to be the top prediction
        exp_logits = np.exp(logits - logits.max())
        output = exp_logits / exp_logits.sum()
        return output.reshape(1, -1)

    def classify(self, bitmap: Image.Image) -> dict:
        """
        Full classification pipeline.
        Returns ClassificationResult equivalent.
        """
        start = time.time()

        # Step 1: Preprocess
        input_tensor = self.preprocess_bitmap(bitmap)

        # Step 2: Inference
        output = self.run_inference(input_tensor)

        # Step 3: Decode
        top_idx = int(np.argmax(output[0]))
        confidence = float(output[0][top_idx])
        disease_name = self.class_names[top_idx]

        inference_ms = (time.time() - start) * 1000

        return {
            'disease_name': disease_name,
            'confidence': confidence,
            'is_healthy': 'healthy' in disease_name.lower(),
            'is_uncertain': confidence < 0.5,
            'inference_ms': inference_ms,
            'top_idx': top_idx
        }


# Test the classifier
classifier = TFLiteClassifierSimulator(CLASS_NAMES)
classifier.initialize()

# Create a test image (simulating a camera capture)
test_leaf = Image.new('RGB', (480, 360), color=(40, 120, 30))
pixels = np.array(test_leaf, dtype=np.uint8)
# Add some brown spots
pixels[100:140, 200:250, :] = [100, 60, 20]
pixels[200:230, 300:340, :] = [110, 70, 25]
test_leaf = Image.fromarray(pixels)

print()
print('Running classification...')
result = classifier.classify(test_leaf)

print()
print('=== CLASSIFICATION RESULT ===')
print(f'  Disease: {result["disease_name"]}')
print(f'  Confidence: {result["confidence"]*100:.1f}%')
print(f'  Healthy: {result["is_healthy"]}')
print(f'  Uncertain: {result["is_uncertain"]}')
print(f'  Inference time: {result["inference_ms"]:.1f} ms')
print()
if result['is_uncertain']:
    print('⚠️  Low confidence — suggest retaking the photo')
elif result['is_healthy']:
    print('✅ Plant is healthy!')
else:
    print(f'⚠️  Disease detected! Treatment needed.')

## Cell 5: Benchmarking TFLite Inference

In [ ]:
import time
import numpy as np
from PIL import Image

try:
    import tensorflow as tf

    # Build a small demo TFLite model
    demo_model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(224, 224, 3)),
        tf.keras.layers.GlobalAveragePooling2D(),
        tf.keras.layers.Dense(38, activation='softmax')
    ])

    converter = tf.lite.TFLiteConverter.from_keras_model(demo_model)
    tflite_model = converter.convert()

    interpreter = tf.lite.Interpreter(model_content=tflite_model)
    interpreter.allocate_tensors()
    input_details = interpreter.get_input_details()[0]
    output_details = interpreter.get_output_details()[0]

    print('=== INFERENCE BENCHMARK ===')
    print()

    # Benchmark: run 100 inferences
    N = 100
    times = []

    for i in range(N):
        test_input = np.random.rand(1, 224, 224, 3).astype(np.float32)
        t0 = time.perf_counter()
        interpreter.set_tensor(input_details['index'], test_input)
        interpreter.invoke()
        _ = interpreter.get_tensor(output_details['index'])
        t1 = time.perf_counter()
        times.append((t1 - t0) * 1000)

    times = np.array(times)
    print(f'Benchmark: {N} inference runs')
    print(f'  Min:    {times.min():.2f} ms')
    print(f'  Max:    {times.max():.2f} ms')
    print(f'  Mean:   {times.mean():.2f} ms')
    print(f'  Median: {np.median(times):.2f} ms')
    print(f'  Std:    {times.std():.2f} ms')
    print()
    print('Expected on Android device (real plant disease model):')
    print('  Mid-range phone CPU:  100-500 ms per inference')
    print('  With GPU delegate:    50-150 ms')
    print('  With NNAPI delegate:  30-100 ms (hardware AI accelerator)')
    print()
    print('Histogram:')
    hist, bins = np.histogram(times, bins=10)
    for i in range(len(hist)):
        bar = '█' * int(hist[i] / max(hist) * 30)
        print(f'  {bins[i]:.1f}-{bins[i+1]:.1f}ms: {bar} ({hist[i]})')

except ImportError:
    print('TensorFlow not installed.')
    print()
    print('To benchmark on your machine:')
    print('  pip install tensorflow')
    print('  Run this cell again')
    print()
    print('Expected inference times for full plant disease model:')
    print("  | Device                     | Time    |")
    print("  |----------------------------|---------|")
    print("  | Laptop CPU (Python TFLite) | 15-50ms |")
    print("  | Mid-range Android CPU      | 100-500ms|")
    print("  | Android + GPU delegate     | 50-150ms|")
    print("  | Android + NNAPI delegate   | 30-100ms|")

## Cell 6: Mapping Python to Android Java

In [ ]:
mapping = [
    {
        'step': 'Load model file',
        'python': '''
# Python TFLite
with open("model/plant_disease.tflite", "rb") as f:
    tflite_model = f.read()
interpreter = tf.lite.Interpreter(model_content=tflite_model)
''',
        'java': '''
// Android Java
private MappedByteBuffer loadModelFile(Context ctx) throws IOException {
    AssetFileDescriptor afd = ctx.getAssets().openFd("plant_disease.tflite");
    FileInputStream fis = new FileInputStream(afd.getFileDescriptor());
    return fis.getChannel().map(
        FileChannel.MapMode.READ_ONLY,
        afd.getStartOffset(), afd.getDeclaredLength()
    );
}
Interpreter interpreter = new Interpreter(loadModelFile(context));
'''
    },
    {
        'step': 'Allocate tensors',
        'python': '''
interpreter.allocate_tensors()
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()
''',
        'java': '''
// Room does this automatically on Interpreter creation
// No explicit allocate_tensors() call needed in Java TFLite API
'''
    },
    {
        'step': 'Preprocess input',
        'python': '''
image = Image.open(io.BytesIO(image_bytes)).convert("RGB")
image = image.resize((224, 224), Image.LANCZOS)
img_array = np.array(image, dtype=np.float32) / 255.0
img_array = np.expand_dims(img_array, axis=0)  # [1,224,224,3]
''',
        'java': '''
Bitmap resized = Bitmap.createScaledBitmap(bitmap, 224, 224, true);
float[][][][] input = new float[1][224][224][3];
for (int y = 0; y < 224; y++) {
    for (int x = 0; x < 224; x++) {
        int pixel = resized.getPixel(x, y);
        input[0][y][x][0] = ((pixel >> 16) & 0xFF) / 255.0f;  // R
        input[0][y][x][1] = ((pixel >> 8)  & 0xFF) / 255.0f;  // G
        input[0][y][x][2] = (pixel          & 0xFF) / 255.0f;  // B
    }
}
'''
    },
    {
        'step': 'Run inference',
        'python': '''
interpreter.set_tensor(input_details[0]['index'], img_array)
interpreter.invoke()
output = interpreter.get_tensor(output_details[0]['index'])
# output shape: (1, 38)
''',
        'java': '''
float[][] output = new float[1][labels.size()];
interpreter.run(input, output);
// output[0] is the 38-element probability array
'''
    },
    {
        'step': 'Decode output',
        'python': '''
top_idx = np.argmax(output[0])
confidence = output[0][top_idx]
disease_name = class_names[top_idx]
''',
        'java': '''
int topIdx = 0;
float topConf = 0f;
for (int i = 0; i < output[0].length; i++) {
    if (output[0][i] > topConf) {
        topConf = output[0][i];
        topIdx = i;
    }
}
return new ClassificationResult(labels.get(topIdx), topConf);
'''
    }
]

print('=== PYTHON TFLite ↔ ANDROID JAVA MAPPING ===')
print()
for item in mapping:
    print(f'STEP: {item["step"]}')
    print()
    print('  Python:', item['python'])
    print('  Java:', item['java'])
    print('─' * 70)

## Cell 7: Checkpoint Exercises

1. ☐ Run the `TFLiteClassifierSimulator` with a different test image
2. ☐ Modify `classify()` to return top-3 predictions instead of just top-1
3. ☐ Add a `confidence_threshold` parameter — return `None` if top confidence < threshold
4. ☐ Open `android-app/.../TFLiteClassifier.java` — map each Java method to its Python equivalent above
5. ☐ (If TF installed) Run the benchmark and compare FP32 vs quantized model speed
6. ☐ Explain: Why does Android need TFLite instead of running Python TensorFlow directly?

**Bonus:** Profile the preprocessing step separately from the inference step. Which is faster? Why?

## Summary

| Concept | Key point |
|---------|----------|
| Why TFLite | Mobile-optimised, 1-2MB runtime, offline capable |
| INT8 quantization | 4× smaller model, 2-4× faster, <1% accuracy loss |
| Conversion | `TFLiteConverter.from_keras_model(model).convert()` |
| Android inference | `Interpreter.run(input, output)` with float[][][][] |
| Preprocessing | Resize → RGB → /255.0 → expand_dims (identical in Python and Java) |
| NNAPI delegate | Routes computation to hardware AI accelerator |

---
*See also: `roadmap/week-09-tensorflow-lite-offline-ai/learning-notes.md`*  
*And: `android-app/app/src/main/java/com/leafguard/ml/TFLiteClassifier.java`*